# Sentiment Analysis — Modeling Notebook

A reproducible baseline pipeline for sentiment-label classification. **Split first, fit TF-IDF on training data only** to prevent test-set leakage. Rare labels are consolidated into `Other` (threshold configurable).

In [ ]:
import re, warnings
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import accuracy_score, classification_report, confusion_matrix
from sklearn.linear_model import LogisticRegression
from sklearn.naive_bayes import MultinomialNB
from sklearn.svm import LinearSVC
from sklearn.ensemble import RandomForestClassifier
import joblib
warnings.filterwarnings('ignore')

In [ ]:
df = pd.read_csv('sentimentdataset.csv')
df['Text'] = df['Text'].astype(str).str.strip()
df['Sentiment'] = df['Sentiment'].astype(str).str.strip()
print(df.shape, 'rows; labels:', df['Sentiment'].nunique())
df.isnull().sum()

In [ ]:
df['Sentiment'].value_counts().head(20).plot(kind='bar', figsize=(12,4), title='Top 20 sentiment labels')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## Text cleaning
These baseline rules remove markup, URLs, numbers, punctuation, and non-alphabetic characters. This may remove meaningful signals such as emoji and punctuation.

In [ ]:
def clean_text(text):
    text = str(text).lower()
    text = re.sub(r'<.*?>', ' ', text)
    text = re.sub(r'https?://\\S+|www\\.\\S+', ' ', text)
    text = re.sub(r'\\d+', ' ', text)
    text = re.sub(r'[^a-z\\s]', ' ', text)
    return re.sub(r'\\s+', ' ', text).strip()
df['Clean_Text'] = df['Text'].apply(clean_text)

## Consolidate rare labels
Labels with fewer than 5 observations are grouped into `Other`. This affects the target definition and should be treated as an experiment.

In [ ]:
MIN_CLASS_COUNT = 5
counts = df['Sentiment'].value_counts()
rare = counts[counts < MIN_CLASS_COUNT].index
df['Model_Label'] = df['Sentiment'].where(~df['Sentiment'].isin(rare), 'Other')
df['Model_Label'].value_counts().head(20)

## Split and fit TF-IDF (leakage-free)

In [ ]:
X_train_text, X_test_text, y_train, y_test = train_test_split(df['Clean_Text'], df['Model_Label'], test_size=0.25, random_state=42, stratify=df['Model_Label'])
tfidf = TfidfVectorizer(max_features=1500, ngram_range=(1,3), min_df=2, max_df=0.95)
X_train = tfidf.fit_transform(X_train_text)
X_test = tfidf.transform(X_test_text)
print(X_train.shape, X_test.shape)

In [ ]:
models = {'Logistic Regression': LogisticRegression(max_iter=1000), 'Multinomial NB': MultinomialNB(), 'Linear SVM': LinearSVC(), 'Random Forest': RandomForestClassifier(n_estimators=200, random_state=42, class_weight='balanced_subsample')}
results=[]
for name, model in models.items():
    model.fit(X_train, y_train)
    pred=model.predict(X_test)
    results.append({'Model':name,'Accuracy':accuracy_score(y_test,pred)})
results_df=pd.DataFrame(results).sort_values('Accuracy',ascending=False)
results_df

In [ ]:
results_df.plot(kind='bar', x='Model', y='Accuracy', legend=False, figsize=(8,4), title='Model comparison'); plt.xticks(rotation=30,ha='right'); plt.tight_layout(); plt.show()

In [ ]:
best_name=results_df.iloc[0]['Model']
best_model=models[best_name]
y_pred=best_model.predict(X_test)
print('Selected:',best_name)
print(classification_report(y_test,y_pred,zero_division=0))

In [ ]:
labels=sorted(pd.unique(y_test))
cm=confusion_matrix(y_test,y_pred,labels=labels)
plt.figure(figsize=(12,10)); sns.heatmap(cm,cmap='Blues',xticklabels=labels,yticklabels=labels)
plt.xlabel('Predicted'); plt.ylabel('Actual'); plt.title('Confusion Matrix'); plt.xticks(rotation=90); plt.tight_layout(); plt.show()

## Save artifacts
Save both artifacts from this corrected experiment together. The app expects these filenames.

In [ ]:
joblib.dump(best_model,'best_sentiment_model.pkl',compress=3)
joblib.dump(tfidf,'tfidf_vectorizer.pkl',compress=3)
print('Artifacts saved')

In [ ]:
examples=['I love this feature!','The service was awful and slow.','It is okay, not too bad but not great either.']
for s in examples:
    print(s, '->', best_model.predict(tfidf.transform([clean_text(s)]))[0])

## Limitations
- Small dataset (732 rows), many fine-grained labels, and uneven class frequency.
- Grouping rare labels changes the classification task.
- Text cleaning may remove useful sentiment cues.
- Single train/test split can produce unstable estimates; consider repeated stratified validation.
- Serialized scikit-learn models are version-sensitive.